## Isolation Forest - Annthyroid Experiment

**Nikolaos Ioannidis**

**Data Wrangling -Individual Assigment 1 - Task 2: Reproducing the Results**

This notebook reproduces the main Isolation Forest experiment on the Annthyroid dataset, based on Liu et al. (2008).

We use scikit-learn to train the model, calculate ROC AUC and examine how random seeds affect the results.

The original paper reports 6,832 observations, while our UCI dataset contains 7,200. This difference is discussed in the report.

## 1. Loading the dataset


In [1]:
import numpy as np

train = np.loadtxt("ann-train.data") # we load and combine the original Annthyroid training and test files.
test = np.loadtxt("ann-test.data")

data = np.vstack((train, test)) 

print("Dataset shape:", data.shape) # we check the total number of observations and columns

Dataset shape: (7200, 22)


## 2. Data preparation

In [2]:
# We keep the six continuous features used in the experiment.
# The first 15 features are binary. The lask column contains class labels.
X = data[:, 15:21]

# We convert the original three classes into binary anomaly labels.
# Classes 1 and 2 = anomaly (1) , class 3 =  normal (0)
y = (data[:, 21] != 3).astype(int)

# We check the number of normal and anomalous observations.
print("Features shape:", X.shape)
print("Normal observations:", np.sum(y == 0))
print("Anomalies:", np.sum(y == 1))

Features shape: (7200, 6)
Normal observations: 6666
Anomalies: 534


## 3. Training the Isolation Forest

In [3]:
from sklearn.ensemble import IsolationForest

# We build 100 isolation trees, each using a random sample of 256 observations.
# We fix the random seed to make this run reproducible.
model = IsolationForest(
    n_estimators=100,
    max_samples=256,
    random_state=0
)

model.fit(X) # we train without using the true anomaly labels (unsupervised).

,n_estimators,100
,max_samples,256
,contamination,'auto'
,max_features,1.0
,bootstrap,False
,n_jobs,None
,random_state,0
,verbose,0
,warm_start,False


## 4. Anomaly scores and evaluation

In [4]:
from sklearn.metrics import roc_auc_score

# we reserve the scores, so higher values indicate stronger anomalies.
scores = -model.score_samples(X)

# we use the true labels only to evaluate anomaly ranking.
auc = roc_auc_score(y, scores)

print(f"ROC AUC: {auc:.4f}")

ROC AUC: 0.7991


## 5. Checking implementation details

In [5]:
import sklearn

# we record the implementation version and main experimental settings.
print("scikit-learn version:", sklearn.__version__)
print("Number of trees:", model.n_estimators)
print("Subsample size:", model.max_samples_)
print("Random state:", model.random_state)
print("Dataset shape:", X.shape)
print("ROC AUC:", round(auc, 4))

scikit-learn version: 1.7.2
Number of trees: 100
Subsample size: 256
Random state: 0
Dataset shape: (7200, 6)
ROC AUC: 0.7991


## 6. Repeating the experiment with different random seeds

In [6]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

aucs = []

for seed in range(10):     # we repeat the experiment with 10 different random seeds.
    model_repeat = IsolationForest(   # we keep the number of trees and subsample size unchanged.
        n_estimators=100,
        max_samples=256,
        random_state=seed
    )

    model_repeat.fit(X)  # we train the model anc calculate its ROC AUC.
    scores_repeat = -model_repeat.score_samples(X)  
    auc_repeat = roc_auc_score(y, scores_repeat)

    aucs.append(auc_repeat)
    print(f"Seed {seed}: AUC = {auc_repeat:.4f}")

print()  # we summarize the variation in performance across runs.
print(f"Mean AUC: {np.mean(aucs):.4f}")
print(f"Standard deviation: {np.std(aucs):.4f}")

Seed 0: AUC = 0.7991
Seed 1: AUC = 0.8262
Seed 2: AUC = 0.8082
Seed 3: AUC = 0.7819
Seed 4: AUC = 0.7902
Seed 5: AUC = 0.7778
Seed 6: AUC = 0.8161
Seed 7: AUC = 0.8032
Seed 8: AUC = 0.7923
Seed 9: AUC = 0.8050

Mean AUC: 0.8000
Standard deviation: 0.0142


## Results and Discussion.

The original paper reports a ROC AUC of 0.82 for the Annthyroid dataset.

Our initial run achieved an AUC of 0.7991.

Across 10 random seeds, the mean AUC was 0.8000, with a standard deviation of 0.0142.

These results are reasonably close to those reported in the paper. The differences may be explained by the larger dataset that we used in our experiment (7,200 observations instead of 6,832), implementation details or randomness in the construction of isolation trees.

Overall, our experiment reproduces the main Isolation Forest procedure and shows that the algorithm can identify anomalous observations without using class labels during training.